# Can a voice model work for a new patient?

Estimate motor UPDRS from 16 voice measurements, then check whether the result survives a split by patient. The target is an interpolated severity score, not a diagnosis or future score.

**How to run:** use Python 3.12; in Colab choose Run all. Installation needs internet. The public UCI data download happens only if `data/parkinsons_updrs.data` is absent. The analysis runs on a CPU and writes tables and figures to `results/`.

Data: Tsanas & Little (2009), [UCI Parkinsons Telemonitoring](https://archive.ics.uci.edu/dataset/189/parkinsons+telemonitoring), [DOI 10.24432/C5ZS3N](https://doi.org/10.24432/C5ZS3N), CC BY 4.0. There are 5,875 recordings from 42 people with early-stage disease.


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *['numpy==2.5.0', 'pandas==3.0.3', 'scikit-learn==1.9.0', 'scipy==1.18.0', 'matplotlib==3.9.2']])


In [1]:
"""Can a voice model work for a new patient?

Run: python SLT.py
Data: UCI Parkinsons Telemonitoring, DOI 10.24432/C5ZS3N, CC BY 4.0.
The target is interpolated motor UPDRS, not a diagnosis or a future score.
"""

from pathlib import Path
import argparse
import hashlib
import io
import json
import platform
import urllib.request
import zipfile

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
DATA_URL = "https://archive.ics.uci.edu/static/public/189/parkinsons+telemonitoring.zip"
FEATURES = [
    "Jitter(%)", "Jitter(Abs)", "Jitter:RAP", "Jitter:PPQ5", "Jitter:DDP",
    "Shimmer", "Shimmer(dB)", "Shimmer:APQ3", "Shimmer:APQ5",
    "Shimmer:APQ11", "Shimmer:DDA", "NHR", "HNR", "RPDE", "DFA", "PPE",
]




## Load the data
The local public table is reused when present. Assertions catch an unexpected schema, missing values or duplicate rows before fitting.

In [2]:
def load_data(data_dir):
    data_dir.mkdir(parents=True, exist_ok=True)
    path = data_dir / "parkinsons_updrs.data"
    if not path.exists():
        with urllib.request.urlopen(DATA_URL, timeout=60) as response:
            archive_bytes = response.read()
        with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
            for name in ["parkinsons_updrs.data", "parkinsons_updrs.names"]:
                (data_dir / name).write_bytes(archive.read(name))
    df = pd.read_csv(path)
    assert df.shape == (5875, 22), "Unexpected dataset shape."
    assert df["subject#"].nunique() == 42, "Unexpected patient count."
    assert not df.isna().any().any(), "Missing values need an explicit policy."
    assert np.isfinite(df.to_numpy()).all(), "Non-finite data found."
    assert not df.duplicated().any(), "Duplicate rows need review."
    return df, hashlib.sha256(path.read_bytes()).hexdigest()




## Define predictors and the patient-level error
Keep patient ID for grouping only. Do not give the model another clinical score. We average errors within a patient before averaging patients, so frequent recorders do not dominate.

In [3]:
def define_task(df):
    return df[FEATURES], df["motor_UPDRS"], df["subject#"].astype(int)


def patient_errors(y, prediction, groups):
    """Give every patient equal weight, regardless of recording count."""
    errors = pd.DataFrame({
        "patient": np.asarray(groups),
        "absolute_error": np.abs(np.asarray(y) - np.asarray(prediction)),
    })
    return errors.groupby("patient")["absolute_error"].mean()


def summarize(y, prediction, groups):
    return {
        "patient_mae": float(patient_errors(y, prediction, groups).mean()),
        "recording_mae": float(mean_absolute_error(y, prediction)),
        "rmse": float(np.sqrt(mean_squared_error(y, prediction))),
        "r2": float(r2_score(y, prediction)),
    }




## Fix the model configurations
Compare a training-mean baseline, ridge and a random forest. The ridge scaler stays inside its pipeline. Settings are fixed before test evaluation.

In [4]:
def candidate_models():
    return {
        "Mean baseline": DummyRegressor(strategy="mean"),
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
        "Random forest": RandomForestRegressor(
            n_estimators=200, min_samples_leaf=5, random_state=SEED, n_jobs=1,
        ),
    }




## Validate on patients the model has not seen
GroupKFold keeps every patient's recordings together. Out-of-fold predictions guide model selection. The bootstrap resamples patients, rather than correlated recordings.

In [5]:
def compare_on_development(X, y, groups, models):
    folds = GroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    rows = []
    for name, model in models.items():
        prediction = np.full(len(y), np.nan)
        for train, valid in folds.split(X, y, groups):
            assert set(groups.iloc[train]).isdisjoint(groups.iloc[valid])
            fitted = clone(model).fit(X.iloc[train], y.iloc[train])
            prediction[valid] = fitted.predict(X.iloc[valid])
        assert np.isfinite(prediction).all()
        rows.append({"model": name, **summarize(y, prediction, groups)})
    return pd.DataFrame(rows).sort_values("patient_mae").reset_index(drop=True)


def run_development_extension(development_df, models, out):
    """Compare voice alone with voice + demographics, using development data only."""
    voice_X, y, groups = define_task(development_df)
    expanded_X = development_df[FEATURES + ["age", "sex"]]
    comparisons = []
    for label, X in [("Voice only", voice_X), ("Voice + age + sex", expanded_X)]:
        cv = compare_on_development(X, y, groups, models)
        cv.insert(0, "feature_set", label)
        comparisons.append(cv)
    comparison = pd.concat(comparisons, ignore_index=True)
    results = {
        "seed": SEED,
        "development_patients": sorted(groups.unique().tolist()),
        "test_evaluated": False,
        "comparison": comparison.to_dict(orient="records"),
    }
    out.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(out / "development_comparison.csv", index=False)
    (out / "metrics.json").write_text(json.dumps(results, indent=2) + "\n")
    print(comparison[["feature_set", "model", "patient_mae"]].round(2).to_string(index=False))
    print(f"\nDevelopment comparison saved to {out}/. Test patients were not evaluated.")
    return results


def bootstrap_patient_mean(errors):
    """Resample patients, not correlated recordings, for a rough 95% interval."""
    values = np.asarray(errors)
    rng = np.random.default_rng(SEED)
    means = rng.choice(values, size=(2000, len(values)), replace=True).mean(axis=1)
    return np.quantile(means, [0.025, 0.975]).tolist()


def save_figures(results, cv, patient_mae, predictions, out):
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
                         "axes.spines.top": False, "axes.spines.right": False})
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(cv["model"], cv["patient_mae"], color=["#c15b3d", "#286777", "#aaa49a"])
    ax.invert_yaxis()
    ax.set_xlabel("Mean patient MAE (motor UPDRS points); lower is better")
    ax.set_title("Model selection: development patients only")
    fig.tight_layout(); fig.savefig(out / "development_comparison.png", dpi=180)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(np.arange(len(patient_mae)), patient_mae.to_numpy(), color="#286777")
    ax.axhline(patient_mae.mean(), color="#c15b3d", linestyle="--", label="Patient mean")
    ax.set(xticks=np.arange(len(patient_mae)),
           xticklabels=[str(v) for v in patient_mae.index],
           xlabel="Held-out patient ID", ylabel="MAE (motor UPDRS points)",
           title="Averages hide differences between patients")
    ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(out / "patient_errors.png", dpi=180)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(predictions["actual"], predictions["predicted"], s=12, alpha=.25,
               color="#286777", edgecolors="none")
    lo = min(predictions[["actual", "predicted"]].min())
    hi = max(predictions[["actual", "predicted"]].max())
    ax.plot([lo, hi], [lo, hi], color="#c15b3d", linestyle="--")
    ax.set(xlabel="Recorded target (interpolated motor UPDRS)",
           ylabel="Predicted motor UPDRS", title="Unseen-patient test predictions")
    fig.tight_layout(); fig.savefig(out / "test_predictions.png", dpi=180)
    plt.close(fig)




## Reserve the test patients
These variables are reused by the next cells. All recordings from a patient stay on one side. Check for 33 development patients, nine test patients and zero overlap.

In [6]:
out = Path("results")
out.mkdir(parents=True, exist_ok=True)
df, checksum = load_data(Path("data"))
X, y, groups = define_task(df)
split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
dev, test = next(split.split(X, y, groups))
assert set(groups.iloc[dev]).isdisjoint(groups.iloc[test])
models = candidate_models()
print(f"Development: {groups.iloc[dev].nunique()} patients, {len(dev)} recordings")
print(f"Test: {groups.iloc[test].nunique()} patients, {len(test)} recordings")
print("Patient overlap:", len(set(groups.iloc[dev]) & set(groups.iloc[test])))


Development: 33 patients, 4640 recordings
Test: 9 patients, 1235 recordings
Patient overlap: 0


## Run grouped validation
This is the call shown on slide 8. Each validation patient is absent from that fold's training data. The helper fits a fresh model in each fold, collects predictions and averages each patient's MAE equally.

For example, if two patients have MAEs of 2 and 8, the patient-average MAE is 5, even if the first patient has many more recordings. The supplied run should show baseline 7.10, ridge 7.52 and forest 7.65. Select using these development scores.

In [7]:
cv = compare_on_development(
    X.iloc[dev], y.iloc[dev], groups.iloc[dev],
    candidate_models()
)
print(cv[["model", "patient_mae"]].round(2).to_string(index=False))


        model  patient_mae
Mean baseline         7.10
        Ridge         7.52
Random forest         7.65


## Fit the selected model and evaluate the reserved patients
Fit the development-selected model on all 33 development patients, then evaluate the nine test patients. This cell does not choose a model from the test results.

In [8]:
selected_name = cv.loc[0, "model"]
selected = clone(models[selected_name]).fit(X.iloc[dev], y.iloc[dev])

prediction = selected.predict(X.iloc[test])
baseline = clone(models["Mean baseline"]).fit(X.iloc[dev], y.iloc[dev])
baseline_prediction = baseline.predict(X.iloc[test])
test_metrics = summarize(y.iloc[test], prediction, groups.iloc[test])
baseline_metrics = summarize(y.iloc[test], baseline_prediction, groups.iloc[test])
errors = patient_errors(y.iloc[test], prediction, groups.iloc[test])
interval = bootstrap_patient_mean(errors)
predictions = pd.DataFrame({
    "row": test, "patient": groups.iloc[test].to_numpy(),
    "actual": y.iloc[test].to_numpy(), "predicted": prediction,
    "baseline_prediction": baseline_prediction,
})


print("Selected:", selected_name)
print(f"Unseen-patient MAE: {test_metrics['patient_mae']:.2f} points")


Selected: Mean baseline
Unseen-patient MAE: 6.90 points


## Compare the split methods and save the original experiment
The fixed-forest recording-split comparison is a separate diagnostic and does not change selection. The two test sets differ, so their error gap is illustrative rather than a controlled estimate of leakage bias. Tables and figures go to results/.

In [9]:
row_train, row_test = train_test_split(np.arange(len(df)), test_size=.20,
                                      random_state=SEED)
forest = clone(models["Random forest"]).fit(X.iloc[dev], y.iloc[dev])
forest_test_metrics = summarize(y.iloc[test], forest.predict(X.iloc[test]),
                                groups.iloc[test])
row_model = clone(models["Random forest"]).fit(X.iloc[row_train], y.iloc[row_train])
row_metrics = summarize(y.iloc[row_test], row_model.predict(X.iloc[row_test]),
                        groups.iloc[row_test])
overlap = sorted(set(groups.iloc[row_train]) & set(groups.iloc[row_test]))

results = {
    "question": "Can voice measurements estimate motor UPDRS for an unseen patient?",
    "seed": SEED, "data_sha256": checksum,
    "recordings": len(df), "patients": int(groups.nunique()),
    "features": FEATURES, "target": "motor_UPDRS (linearly interpolated)",
    "development_rows": len(dev), "test_rows": len(test),
    "development_patients": sorted(groups.iloc[dev].unique().tolist()),
    "test_patients": sorted(groups.iloc[test].unique().tolist()),
    "selected_model": selected_name,
    "development_cv": cv.to_dict(orient="records"),
    "test": test_metrics, "test_baseline": baseline_metrics,
    "fixed_forest_test": forest_test_metrics,
    "patient_mae_95pct_bootstrap_interval": interval,
    "patient_errors": [{"patient": int(k), "mae": float(v)} for k, v in errors.items()],
    "random_recording_split": {**row_metrics, "overlapping_patients": overlap,
                                "test_rows": len(row_test)},
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
                 "matplotlib": matplotlib.__version__},
}
(out / "metrics.json").write_text(json.dumps(results, indent=2) + "\n")
cv.to_csv(out / "development_comparison.csv", index=False)
predictions.to_csv(out / "test_predictions.csv", index=False)
errors.rename("mae").to_csv(out / "patient_errors.csv")
pd.DataFrame({"row": np.arange(len(df)), "patient": groups,
              "split": np.where(np.isin(np.arange(len(df)), test), "test", "development")
              }).to_csv(out / "split_membership.csv", index=False)
save_figures(results, cv, errors, predictions, out)
print(cv[["model", "patient_mae"]].to_string(index=False))
print(f"\nSelected: {selected_name}")
print(f"Unseen-patient MAE: {test_metrics['patient_mae']:.2f} points")
print(f"Same-test baseline MAE: {baseline_metrics['patient_mae']:.2f} points")
print(f"Patient bootstrap 95% interval: {interval[0]:.2f} to {interval[1]:.2f}")
print(f"Fixed forest, unseen-patient MAE: {forest_test_metrics['patient_mae']:.2f} points")
print(f"Fixed forest, random-recording MAE: {row_metrics['patient_mae']:.2f} points")
print(f"Patient overlap: group split 0; recording split {len(overlap)}")


        model  patient_mae
Mean baseline     7.096519
        Ridge     7.516904
Random forest     7.649150

Selected: Mean baseline
Unseen-patient MAE: 6.90 points
Same-test baseline MAE: 6.90 points
Patient bootstrap 95% interval: 4.99 to 8.82
Fixed forest, unseen-patient MAE: 6.77 points
Fixed forest, random-recording MAE: 5.21 points
Patient overlap: group split 0; recording split 42


## Read the result before changing anything

In the supplied run, the mean baseline wins grouped development validation (7.10 points; ridge 7.52; forest 7.65). Its held-out patient MAE is 6.90, with a rough patient-bootstrap interval of 4.99–8.82. The fixed forest gives 6.77 on unseen patients versus 5.21 with overlapping patients. The small forest advantage on the nine-patient test does not override development-based selection.

These are severity estimates within this cohort. The interpolated labels, small patient sample and lack of healthy controls prevent claims about diagnosis, future progression or clinical readiness.

The next cell adds age and sex in a separate development-only experiment. It receives only development rows and uses the same patient folds for both feature sets. Keep the original test results as the original experiment. Do not choose changes by repeatedly checking these nine test patients.

**Checkpoint:** why would `total_UPDRS` be an inappropriate input for a voice-only model of motor UPDRS? It provides overlapping clinical outcome information, rather than voice evidence.

Methods: [grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data), [data leakage](https://scikit-learn.org/stable/common_pitfalls.html), [MAE](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html).

Original study: Tsanas et al. (2010), *IEEE TBME*, 57(4), 884–893. [DOI 10.1109/TBME.2009.2036000](https://doi.org/10.1109/TBME.2009.2036000). This notebook uses the dataset rather than reproducing that study's original modeling protocol.


## Extension: voice plus age and sex
Run this cell to compare both feature sets on development data only. It saves a six-row comparison to results_extension/, leaving results/ unchanged. Compare each model's two MAEs. From the terminal, the equivalent command is `python SLT.py --development-only`.

In [10]:
extension = run_development_extension(
    df.iloc[dev].copy(), candidate_models(), Path("results_extension")
)


      feature_set         model  patient_mae
       Voice only Mean baseline         7.10
       Voice only         Ridge         7.52
       Voice only Random forest         7.65
Voice + age + sex Mean baseline         7.10
Voice + age + sex         Ridge         7.24
Voice + age + sex Random forest         8.90

Development comparison saved to results_extension/. Test patients were not evaluated.
